# Differentiable Renderer Analysis

This notebook demonstrates Finite Difference gradient visualizations and advanced optimization timelapses.

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
import gc
import pandas as pd

from PIL import Image
import numpy as np
import imageio
from IPython.display import display, Image
# from PIL import Image


diff_render_dir ='diff_render/'
if diff_render_dir not in sys.path:
    sys.path.insert(0, diff_render_dir)

from scene_parser import load_scene_from_xml
from path import PathTracer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def render_crn(scene, cam, integrator, seed=42, requires_grad=False):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if requires_grad:
        return integrator.sample(scene, cam)
    with torch.no_grad():
        return integrator.sample(scene, cam)

def plot_fd(img_base, img_pert, fd_grad, title_pert, h_val, vmin=-25, vmax=25):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(np.clip(img_base, 0, 1))
    axes[0].set_title("Base Render f(x)", fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(np.clip(img_pert, 0, 1))
    axes[1].set_title(f"Perturbed Render f(x + h)\n({title_pert} h={h_val})", fontsize=12)
    axes[1].axis('off')
    
    im2 = axes[2].imshow(fd_grad, cmap='coolwarm', vmin=vmin, vmax=vmax)
    axes[2].set_title("FD Gradient Map\n[f(x+h) - f(x)] / h", fontsize=12)
    axes[2].axis('off')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()


## 1. Teapot: X-Shift Finite Differences

In [ ]:
scene_name = "teapot"
scene_path = f'scenes/{scene_name}/{scene_name}.xml'
if not os.path.exists(scene_path): scene_path = f'scenes/{scene_name}/scene.xml'
scene, cam, _ = load_scene_from_xml(scene_path, device=device, override_res=1024)

integrator_fd = PathTracer(max_depth=4, num_samples=128)

# Ensure output directory exists
out_dir = f"renders/{scene_name}"
os.makedirs(out_dir, exist_ok=True)

img_base = render_crn(scene, cam, integrator_fd).cpu().numpy()
imageio.imwrite(os.path.join(out_dir, f'{scene_name}_base.png'), (np.clip(img_base ** (1/2.2), 0, 1) * 255).astype(np.uint8))

display(Image(os.path.join(out_dir, f'{scene_name}_base.png')))

In [ ]:
scene_name = "teapot"
mesh = scene.get_mesh("teapot")
h = 0.01
perturbation_type = "x_shift"

# Perturb and render
mesh.translate([h, 0.0, 0.0])
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert translation
mesh.translate([-h, 0.0, 0.0])

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

# Save standard renders
imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-25, vmax=25)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "X-shift", h, vmin=-25, vmax=25)

## 2. Teapot: Roughness Shift Finite Differences

In [ ]:
scene_name = "teapot" 
mesh = scene.get_mesh("teapot")

h = 0.1
perturbation_type = "roughness"

# Perturb and render
mesh.roughness += torch.tensor([h], device=device)
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert roughness
mesh.roughness -= torch.tensor([h], device=device)

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map (note the updated vmin/vmax for roughness)
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-5, vmax=5)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "Roughness", h, vmin=-5, vmax=5)

## 3. Bunny (cbox): X-Shift Finite Differences

In [ ]:
scene_name = "cbox_bunny"
scene_path = 'scenes/cbox/cbox_bunny.xml'
if not os.path.exists(scene_path): scene_path = 'scenes/cbox/scene.xml'
scene, cam, _ = load_scene_from_xml(scene_path, device=device, override_res=1024)

out_dir = f"renders/{scene_name}"
os.makedirs(out_dir, exist_ok=True)

img_base = render_crn(scene, cam, integrator_fd).cpu().numpy()
imageio.imwrite(os.path.join(out_dir, f'{scene_name}_base.png'), (np.clip(img_base ** (1/2.2), 0, 1) * 255).astype(np.uint8))

display(Image(os.path.join(out_dir, f'{scene_name}_base.png')))

In [ ]:
mesh = scene.get_mesh("bunny")

h = 0.01
perturbation_type = "x_shift"

# Perturb and render
mesh.translate([h, 0.0, 0.0])
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert translation
mesh.translate([-h, 0.0, 0.0])

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-25, vmax=25)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "X-shift", h, vmin=-25, vmax=25)

## 4. Bunny (cbox): Albedo Shift Finite Differences

In [ ]:
mesh = scene.get_mesh("bunny")

h = 0.1
perturbation_type = "albedo_red"

# Perturb and render
mesh.albedo += torch.tensor([h, 0.0, 0.0], device=device)
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert albedo
mesh.albedo -= torch.tensor([h, 0.0, 0.0], device=device)

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-0.5, vmax=0.5)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "Albedo", h, vmin=-0.5, vmax=0.5)

## 5. Box Cubes: Red Cube X-Shift Finite Differences

In [ ]:
scene_name = "box_cubes"
scene_path = 'scenes/box_cubes/box_cubes.xml'

scene, cam, _ = load_scene_from_xml(scene_path, device=device, override_res=1024)

out_dir = f"renders/{scene_name}"
os.makedirs(out_dir, exist_ok=True)

img_base = render_crn(scene, cam, integrator_fd).cpu().numpy()
imageio.imwrite(os.path.join(out_dir, f'{scene_name}_base.png'), (np.clip(img_base ** (1/2.2), 0, 1) * 255).astype(np.uint8))

display(Image(os.path.join(out_dir, f'{scene_name}_base.png')))
out_dir

In [ ]:
mesh = scene.get_mesh("red_cube")

h = 0.01
perturbation_type = "x_shift"

# Perturb and render
mesh.translate([h, 0.0, 0.0])
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert translation
mesh.translate([-h, 0.0, 0.0])

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-10, vmax=10)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "X-shift", h, vmin=-10, vmax=10)

## 6. Box Cubes: Red Cube Roughness Finite Differences

In [ ]:
mesh = scene.get_mesh("floor")

h_roughness = 0.1
perturbation_type = "floor_roughness"

# Perturb and render
mesh.roughness += h_roughness
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert roughness
mesh.roughness -= h_roughness

fd_grad = np.mean((img_pert - img_base) / h_roughness, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_h_{h_roughness}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_h_{h_roughness}.png'), fd_grad, cmap='coolwarm', vmin=-10, vmax=10)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, "Roughness (Floor)", h_roughness, vmin=-10, vmax=10)

## 7. Shadow Glossy: X-Shift Finite Differences

In [ ]:
scene_name = "shadow_glossy"
scene_path = 'scenes/shadow_glossy/shadow_glossy.xml'

scene, cam, _ = load_scene_from_xml(scene_path, device=device, override_res=1024)

out_dir = f"renders/{scene_name}"
os.makedirs(out_dir, exist_ok=True)

img_base = render_crn(scene, cam, integrator_fd).cpu().numpy()
imageio.imwrite(os.path.join(out_dir, f'{scene_name}_base.png'), (np.clip(img_base ** (1/2.2), 0, 1) * 255).astype(np.uint8))

display(Image(os.path.join(out_dir, f'{scene_name}_base.png')))

In [ ]:
out_dir = f"renders/{scene_name}"
os.makedirs(out_dir, exist_ok=True)

mesh_name = list(scene.mesh_map.keys())[1]
mesh = scene.get_mesh(mesh_name)

h = 0.01
perturbation_type = f"{mesh_name}_x_shift"

# Perturb and render
mesh.translate([h, 0.0, 0.0])
img_pert = render_crn(scene, cam, integrator_fd).cpu().numpy()
# Revert translation
mesh.translate([-h, 0.0, 0.0])

fd_grad = np.mean((img_pert - img_base) / h, axis=-1)

imageio.imwrite(os.path.join(out_dir, f'{scene_name}_pert_{perturbation_type}_h_{h}.png'), (np.clip(img_pert ** (1/2.2), 0, 1) * 255).astype(np.uint8))

# Save gradient map
plt.imsave(os.path.join(out_dir, f'{scene_name}_fd_grad_{perturbation_type}_h_{h}.png'), fd_grad, cmap='coolwarm', vmin=-25, vmax=25)

plot_fd(img_base ** (1/2.2), img_pert ** (1/2.2), fd_grad, f"X-shift ({mesh_name})", h, vmin=-25, vmax=25)

## 8. Extreme Texture Optimization (Planets)

Adapted from `optimize_texture_extreme.py`.

In [ ]:
from diff_render.prb import PRBPathTracer

res = 1024
iters = 20 # Kept low for notebook execution speed. Change to 200 for extreme visualization.
scene_mars, cam, _ = load_scene_from_xml("scenes/planets/mars.xml", device=device, override_res=res)
scene_earth, _, _ = load_scene_from_xml("scenes/planets/earth.xml", device=device, override_res=res)

# Swap to PRB Path Tracer
prb_tracer = PRBPathTracer(max_depth=4, num_samples=128)

print("Rendering Target and Initial Images...")
with torch.no_grad():
    img_target = prb_tracer.sample_path(scene_earth, cam, seed=42).detach()
    img_initial = prb_tracer.sample_path(scene_mars, cam, seed=42).detach()

tex_var = scene_mars.textures[0]
tex_var.requires_grad_(True)
tex_target = scene_earth.textures[0]

# 1. Setup directories FIRST before saving anything
out_dir = "renders/planets/"
frames_render_dir = os.path.join(out_dir, "frames_render")
frames_tex_dir = os.path.join(out_dir, "frames_tex")

os.makedirs(out_dir, exist_ok=True)
os.makedirs(frames_render_dir, exist_ok=True)
os.makedirs(frames_tex_dir, exist_ok=True)

# 2. Save initial and target states
plt.imsave(os.path.join(out_dir, "initial_render.png"), np.clip(img_initial.cpu().numpy(), 0, 1))
plt.imsave(os.path.join(out_dir, "target_render.png"), np.clip(img_target.cpu().numpy(), 0, 1))

plt.imsave(os.path.join(out_dir, "initial_texture.png"), np.clip(tex_var.detach().cpu().numpy(), 0, 1))
plt.imsave(os.path.join(out_dir, "target_texture.png"), np.clip(tex_target.detach().cpu().numpy(), 0, 1))

optimizer = torch.optim.Adam([tex_var], lr=0.1)

frames_render = []
frames_tex = []

print("Starting PRB optimization loop...")
for i in range(iters):
    optimizer.zero_grad()
    
    # Forward pass using PRB
    img_pred = prb_tracer.sample_path(scene_mars, cam, seed=42)
    
    # Compute loss for logging
    loss = F.mse_loss(img_pred, img_target)
    
    # Backward pass using PRB
    # The derivative of mean((pred - target)^2) wrt pred is 2 * (pred - target) / N
    dL_dimg = 2.0 * (img_pred - img_target) / img_pred.numel()
    prb_tracer.sample_adjoint(scene_mars, cam, img_pred, dL_dimg)
    
    optimizer.step()
    
    with torch.no_grad():
        tex_var.clamp_(0.0, 1.0)
        
    print(f"Iter {i:03d} | Loss: {loss.item():.4f}")
    
    # Process frames
    frame_r = np.clip(img_pred.detach().cpu().numpy(), 0, 1)
    frame_t = np.clip(tex_var.detach().cpu().numpy(), 0, 1)
    
    frame_r_uint8 = (frame_r * 255).astype(np.uint8)
    frame_t_uint8 = (frame_t * 255).astype(np.uint8)
    
    # Append for the GIF
    frames_render.append(frame_r_uint8)
    frames_tex.append(frame_t_uint8)
    
    # Save individual frames to their respective subfolders
    imageio.imwrite(os.path.join(frames_render_dir, f"frame_{i:03d}.png"), frame_r_uint8)
    imageio.imwrite(os.path.join(frames_tex_dir, f"frame_{i:03d}.png"), frame_t_uint8)


# 3. Save the GIFs
render_gif_path = os.path.join(out_dir, "render_timelapse.gif")
tex_gif_path = os.path.join(out_dir, "texture_timelapse.gif")

imageio.mimsave(render_gif_path, frames_render, fps=20)
imageio.mimsave(tex_gif_path, frames_tex, fps=20)

print(f"Optimization complete. Individual frames and GIFs saved to {out_dir}!")

In [ ]:
out_dir = "renders/planets/"
os.makedirs(out_dir, exist_ok=True)
render_gif_path = os.path.join(out_dir, "render_timelapse.gif")
tex_gif_path = os.path.join(out_dir, "texture_timelapse.gif")

display(Image(filename=render_gif_path))
display(Image(filename=tex_gif_path))


## 9. Normal Map Optimization (Steps)

Adapted from `optimize_moon.py` and `optimize_normal.py`.

In [ ]:
from diff_render.prb import PRBPathTracer

res = 1024
iters = 20 
scene_init, cam, _ = load_scene_from_xml("scenes/steps/scene.xml", device=device, override_res=res)
scene_target, _, _ = load_scene_from_xml("scenes/steps/scene_target.xml", device=device, override_res=res)

# Swap to PRB Path Tracer
prb_tracer = PRBPathTracer(max_depth=3, num_samples=128)

print("Rendering Target and Initial Images...")
with torch.no_grad():
    img_target = prb_tracer.sample_path(scene_target, cam, seed=42).detach()
    img_initial = prb_tracer.sample_path(scene_init, cam, seed=42).detach()

# Fix path and setup directories for frames
out_dir = "renders/steps"
frames_render_dir = os.path.join(out_dir, "frames_render")
frames_norm_dir = os.path.join(out_dir, "frames_norm")

os.makedirs(out_dir, exist_ok=True)
os.makedirs(frames_render_dir, exist_ok=True)
os.makedirs(frames_norm_dir, exist_ok=True)

plt.imsave(os.path.join(out_dir, "initial_render.png"), np.clip(img_initial.cpu().numpy(), 0, 1))
plt.imsave(os.path.join(out_dir, "target_render.png"), np.clip(img_target.cpu().numpy(), 0, 1))

nmap_var = scene_init.normal_maps[0]
nmap_var.requires_grad_(True)
nmap_target = scene_target.normal_maps[0]

plt.imsave(os.path.join(out_dir, "initial_normal.png"), np.clip(nmap_var.detach().cpu().numpy(), 0, 1))
plt.imsave(os.path.join(out_dir, "target_normal.png"), np.clip(nmap_target.detach().cpu().numpy(), 0, 1))

# Optimizing ONLY the normal map tensor
optimizer = torch.optim.Adam([nmap_var], lr=0.05)

frames_render = []
frames_norm = []

print("Starting PRB Normal Map Optimization Loop...")
for i in range(iters):
    optimizer.zero_grad()
    
    # Forward pass using PRB
    img_pred = prb_tracer.sample_path(scene_init, cam, seed=42)
    
    # Compute loss for logging
    loss = F.mse_loss(img_pred, img_target)
    
    # Backward pass using PRB
    dL_dimg = 2.0 * (img_pred - img_target) / img_pred.numel()
    prb_tracer.sample_adjoint(scene_init, cam, img_pred, dL_dimg)

    torch.nan_to_num_(nmap_var.grad, nan=0.0, posinf=0.0, neginf=0.0)
    torch.nn.utils.clip_grad_norm_([nmap_var], max_norm=1.0)
    optimizer.step()
    
    with torch.no_grad():
        nmap_var.clamp_(0.0, 1.0)
        nmap_var[:, :, 2].clamp_(0.5, 1.0)
        
    print(f"Iter {i:03d} | Loss: {loss.item():.4f}")
    
    frame_r = np.clip(img_pred.detach().cpu().numpy(), 0, 1)
    frame_n = np.clip(nmap_var.detach().cpu().numpy(), 0, 1)
    
    frame_r_uint8 = (frame_r * 255).astype(np.uint8)
    frame_n_uint8 = (frame_n * 255).astype(np.uint8)
    
    frames_render.append(frame_r_uint8)
    frames_norm.append(frame_n_uint8)
    
    # Save individual frames
    imageio.imwrite(os.path.join(frames_render_dir, f"frame_{i:03d}.png"), frame_r_uint8)
    imageio.imwrite(os.path.join(frames_norm_dir, f"frame_{i:03d}.png"), frame_n_uint8)
    
plt.imsave(os.path.join(out_dir, "final_render.png"), np.clip(img_pred.detach().cpu().numpy(), 0, 1))
plt.imsave(os.path.join(out_dir, "final_normal.png"), np.clip(nmap_var.detach().cpu().numpy(), 0, 1))

print("Saving optimization GIFs...")
imageio.mimsave(os.path.join(out_dir, "render_timelapse.gif"), frames_render, fps=15)
imageio.mimsave(os.path.join(out_dir, "normal_timelapse.gif"), frames_norm, fps=15)

In [ ]:
out_dir = "renders/steps/"
os.makedirs(out_dir, exist_ok=True)
render_gif_path = os.path.join(out_dir, "render_timelapse.gif")
tex_gif_path = os.path.join(out_dir, "normal_timelapse.gif")


display(Image(filename=render_gif_path))
display(Image(filename=tex_gif_path))


## 10. Gradient Comparison: Naive AD vs PRB vs RB

Here we evaluate the exact gradient vectors computed by Naive AD, Path Replay Backpropagation (PRB), and Radiative Backpropagation (RB) for the red cube's albedo. 

We also compute the finite difference directional derivative to visualize the sensitivity of the image pixels to the albedo's gradient vector in image space. As mathematically expected for an interior property like albedo, Naive AD and PRB perfectly agree when tracing identical CRN paths in a purely linear space.

In [ ]:
from path import PathTracer
from prb import PRBPathTracer
from rb import RBPathTracer

# --- Helper Functions ---
def compute_heatmap(img_tensor):
    val = img_tensor.sum(dim=-1)
    val = val.reshape(512, 512).cpu().numpy()
    vmax = np.abs(val).max()
    return val, vmax

def run_finite_difference(scene, cam, tracer, param_tensor, direction, eps=1e-3, seed=42):
    with torch.no_grad():
        # + eps
        param_tensor.copy_(param_tensor + eps * direction)
        torch.manual_seed(seed)
        img_plus = tracer.sample(scene, cam) if hasattr(tracer, 'sample') else tracer.sample_path(scene, cam, seed=seed)
        
        # - eps
        param_tensor.copy_(param_tensor - 2 * eps * direction)
        torch.manual_seed(seed)
        img_minus = tracer.sample(scene, cam) if hasattr(tracer, 'sample') else tracer.sample_path(scene, cam, seed=seed)
        
        # Restore original parameter
        param_tensor.copy_(param_tensor + eps * direction)
        
    return (img_plus - img_minus) / (2 * eps)

# --- Setup Scene ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
scene_name = "box_cubes"

# Handle paths depending on where the notebook is located
scene_path = f'scenes/{scene_name}/{scene_name}.xml'
if not os.path.exists(scene_path): 
    scene_path = f'scenes/{scene_name}/scene.xml'
    
scene, cam, _ = load_scene_from_xml(scene_path, device=device, override_res=512)

red_cube = scene.get_mesh("red_cube")
albedo_param = red_cube.albedo.detach().clone().requires_grad_(True)
red_cube.albedo = albedo_param

num_samples = 128
path_tracer = PathTracer(max_depth=3, num_samples=num_samples)
prb_tracer = PRBPathTracer(max_depth=3, num_samples=num_samples)
rb_tracer = RBPathTracer(max_depth=3, num_samples=num_samples)

# --- Compute Gradients ---
print(f"--- Computing Gradients at {num_samples} SPP ---")

# 1. Naive AD
torch.manual_seed(42)
img_naive = path_tracer.sample(scene, cam)
loss_naive = (img_naive ** 2).sum()
loss_naive.backward()
grad_naive = albedo_param.grad.clone()
albedo_param.grad.zero_()

# 2. PRB
img_prb = prb_tracer.sample_path(scene, cam, seed=42)
dL_prb = 2.0 * img_prb
prb_tracer.sample_adjoint(scene, cam, img_prb, dL_prb)
grad_prb = albedo_param.grad.clone()
albedo_param.grad.zero_()

# 3. RB
img_rb = rb_tracer.sample_path(scene, cam, seed=42)
dL_rb = 2.0 * img_rb
rb_tracer.radiative_backprop(scene, cam, dL_rb)
grad_rb = albedo_param.grad.clone()
albedo_param.grad.zero_()

# --- Verify Matches ---
print(f"Naive AD gradient norm: {torch.norm(grad_naive).item():.6f}")
print(f"PRB gradient norm:      {torch.norm(grad_prb).item():.6f}")
print(f"RB gradient norm:       {torch.norm(grad_rb).item():.6f}")

print(f"\nDifference (Naive vs PRB): {torch.norm(grad_naive - grad_prb).item():.6f}")
print(f"Difference (Naive vs RB):  {torch.norm(grad_naive - grad_rb).item():.6f}")